In [1]:
import pandas as pd
from google.cloud import storage

BUCKET = "filipburger-flight-data-lake"
PREFIX = "raw/bts_lookups"


def load_lookups(bucket: str = BUCKET, prefix: str = PREFIX) -> dict[str, pd.DataFrame]:
    """Load every lookup parquet from GCS into a dict keyed by table name."""
    client = storage.Client()
    frames = {}
    for blob in client.list_blobs(bucket, prefix=prefix):
        if not blob.name.endswith(".parquet"):
            continue
        table = blob.name.split("/")[-2]        # raw/bts_lookups/<table>/data.parquet
        frames[table] = pd.read_parquet(f"gs://{bucket}/{blob.name}")
    return frames


lookups = load_lookups()
print(f"Loaded {len(lookups)} tables")

Loaded 16 tables


In [20]:
summary = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "cols": len(df.columns),
        "columns": ", ".join(c for c in df.columns if not c.startswith("_")),
    }
    for name, df in sorted(lookups.items())
])
summary

,table,rows,cols,columns
0,l_airline_id,1778,3,"Code, Description"
1,l_cancellation,4,3,"Code, Description"
2,l_carrier_history,2075,3,"Code, Description"
3,l_city_market_id,6181,3,"Code, Description"
4,l_deparrblk,19,3,"Code, Description"
5,l_distance_group_250,11,3,"Code, Description"
6,l_diversions,7,3,"Code, Description"
7,l_months,12,3,"Code, Description"
8,l_ontime_delay_groups,15,3,"Code, Description"
9,l_quarters,4,3,"Code, Description"


In [21]:
for name, df in sorted(lookups.items()):
    print(f"\n{'─' * 70}\n{name}  ({len(df):,} rows)")
    print(df.drop(columns=[c for c in df.columns if c.startswith("_")]).head(3).to_string(index=False))


──────────────────────────────────────────────────────────────────────
l_airline_id  (1,778 rows)
 Code                     Description
19031  Mackey International Inc.: MAC
19032 Munz Northern Airlines Inc.: XY
19033      Cochise Airlines Inc.: COC

──────────────────────────────────────────────────────────────────────
l_cancellation  (4 rows)
Code         Description
   A             Carrier
   B             Weather
   C National Air System

──────────────────────────────────────────────────────────────────────
l_carrier_history  (2,075 rows)
Code                       Description
 02Q           Titan Airways (2006 - )
 04Q  Tradewind Aviation (2006 - 2023)
 05Q Comlux Aviation, AG (2006 - 2012)

──────────────────────────────────────────────────────────────────────
l_city_market_id  (6,181 rows)
 Code          Description
30001     Afognak Lake, AK
30003 Granite Mountain, AK
30004              Lik, AK

──────────────────────────────────────────────────────────────────────
l_deparrb

We ned only:
- l_unique_carriers table join IATA_CODE_Reporting_Airline on .Code, to get carrier name
hidden in .Description

- l_cancellation table to join on CancellationCode to get cancellation cause in Description

- l_ontime_delay_groups to join DepartureDelayGroups, ArrivalDelayGroups

In [28]:
iatas =["9E",
"AA",
"AS",
"B6",
"DL",
"F9",
"G4",
"HA",
"MQ",
"NK",
"OH",
"OO",
"UA",
"WN",
"YX"]

In [21]:
ids = ["20363",
"19805",
"19930",
"20409",
"19790",
"20436",
"20368",
"19690",
"20398",
"20416",
"20397",
"20304",
"19977",
"19393",
"20452"]


In [23]:
airline_id = lookups['l_airline_id']
airline_id_filtered = airline_id[(airline_id['Code'].isin(ids))]

In [24]:
airline_id_filtered

,Code,Description,_ingested_at
362,19393,Southwest Airlines Co.: WN,2026-08-16 09:35:14.815304+00:00
659,19690,Hawaiian Airlines Inc.: HA,2026-08-16 09:35:14.815304+00:00
758,19790,Delta Air Lines Inc.: DL,2026-08-16 09:35:14.815304+00:00
773,19805,American Airlines Inc.: AA,2026-08-16 09:35:14.815304+00:00
898,19930,Alaska Airlines Inc.: AS,2026-08-16 09:35:14.815304+00:00
945,19977,United Air Lines Inc.: UA,2026-08-16 09:35:14.815304+00:00
1271,20304,SkyWest Airlines Inc.: OO,2026-08-16 09:35:14.815304+00:00
1330,20363,Endeavor Air Inc.: 9E,2026-08-16 09:35:14.815304+00:00
1335,20368,Allegiant Air: G4,2026-08-16 09:35:14.815304+00:00
1363,20397,PSA Airlines Inc.: OH,2026-08-16 09:35:14.815304+00:00


In [10]:
airline_id.info()

<class 'pandas.DataFrame'>
RangeIndex: 1778 entries, 0 to 1777
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   Code          1778 non-null   string             
 1   Description   1778 non-null   string             
 2   _ingested_at  1778 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), string(2)
memory usage: 94.8 KB


In [24]:

airline_id[airline_id['Code']== '19393']

,Code,Description,_ingested_at
362,19393,Southwest Airlines Co.: WN,2026-08-16 09:35:14.815304+00:00


In [25]:
carriers = lookups['l_unique_carriers']

In [26]:
carriers[carriers['Code']=='WN']

,Code,Description,_ingested_at
1652,WN,Southwest Airlines Co.,2026-08-16 09:35:14.595406+00:00


In [26]:
carrier_history = lookups['l_carrier_history']
carrier_history.head(30)

,Code,Description,_ingested_at
0,02Q,Titan Airways (2006 - ),2026-08-16 09:35:14.925882+00:00
1,04Q,Tradewind Aviation (2006 - 2023),2026-08-16 09:35:14.925882+00:00
2,05Q,"Comlux Aviation, AG (2006 - 2012)",2026-08-16 09:35:14.925882+00:00
3,06Q,Master Top Linhas Aereas Ltd. (2007 - ),2026-08-16 09:35:14.925882+00:00
4,07Q,Flair Airlines Ltd. (2007 - ),2026-08-16 09:35:14.925882+00:00
5,09Q,"Swift Air, LLC (2006 - 2017)",2026-08-16 09:35:14.925882+00:00
6,09Q,"Swift Air, LLC d/b/a Eastern Air Lines d/b/a E...",2026-08-16 09:35:14.925882+00:00
7,0BQ,DCA (2007 - ),2026-08-16 09:35:14.925882+00:00
8,0CQ,ACM AIR CHARTER GmbH (2007 - ),2026-08-16 09:35:14.925882+00:00
9,0FQ,"Maine Aviation Aircraft Charter, LLC (2017 - )",2026-08-16 09:35:14.925882+00:00


In [30]:
filtered_carrier_history = carrier_history[carrier_history['Code'].isin(iatas)]

In [31]:
filtered_carrier_history

,Code,Description,_ingested_at
208,9E,Endeavor Air Inc. (2013 - ),2026-08-16 09:35:14.925882+00:00
209,9E,Pinnacle Airlines Inc. (2002 - 2013),2026-08-16 09:35:14.925882+00:00
231,AA,American Airlines Inc. (1960 - ),2026-08-16 09:35:14.925882+00:00
375,AS,Alaska Airlines Inc. (1960 - ),2026-08-16 09:35:14.925882+00:00
425,B6,JetBlue Airways (2000 - ),2026-08-16 09:35:14.925882+00:00
650,DL,Delta Air Lines Inc. (1960 - ),2026-08-16 09:35:14.925882+00:00
753,F9,Frontier Airlines Inc. (1994 - ),2026-08-16 09:35:14.925882+00:00
807,G4,Allegiant Air (2000 - ),2026-08-16 09:35:14.925882+00:00
885,HA,Hawaiian Airlines Inc. (1960 - 2025),2026-08-16 09:35:14.925882+00:00
1211,MQ,Simmons Airlines (1991 - 1998),2026-08-16 09:35:14.925882+00:00
